In [7]:
import requests
from bs4 import BeautifulSoup
from typing import Dict, Any, List
import time
from tqdm import tqdm 
import json

In [8]:
import VOATibetan_utils

## with the help of VOATibetan_utils.py we can use two main function
1. **extract_all_VOATibetan_article_links: Extracts all article links from a given VOATibetan_utils webpage.**


------------
-----------
------------
------------
------------

In [9]:
def loop_article_page(total_page, custom_url, key_code):
    """
    Loops through article pages on VOA Tibetan website and extracts article links.
    
    Args:
        total_page (int): Maximum number of pages to scrape, used as a safeguard
        custom_url (str): Base URL for the VOA Tibetan articles
        key_code (str): Prefix for the keys in the output dictionary
        
    Returns:
        dict: Dictionary containing scraped data, success message, and response code
    """
    
    return_file = {
        "Data": [],
        "message": "success",
        "response": 200
    }
    All_url_links = {}
    pbar = tqdm(total=total_page)

    try:
        page_index = 0  # Start from page 1 instead of 98
        key_index = 0
        pages_processed = 0
        
        while pages_processed < total_page:  # Use total_page parameter as a limit
            pbar.update(1)
            final_url = custom_url + str(page_index)
            try: 
                found_url_links, load_more, date = VOATibetan_utils.extract_all_VOATibetan_article_links(final_url)
                
                # Store the found links
                key = key_code + str(page_index) + "_" + str(key_index)
                All_url_links[key] = found_url_links
                
                # Check if we need to load more pages or change the date
                if not load_more:
                    if found_url_links["Links"] and len(date) == 3:
                        # Update URL to next date range
                        custom_url = "https://www.voatibetan.com/z/2253"
                        date_url = f"/{date[0]}/{date[1]}/{date[2]}?p="
                        custom_url = custom_url + date_url
                        print(f"Moving to new date range: {custom_url}")
                        page_index = 0  # Reset page index for new date
                    else:
                        # No more pages to load and no new date
                        print(f"Final page number: {page_index}")
                        print(f"Total pages extracted: {key_index}")
                        if len(date) == 3:
                            print(f"Last date: {date[0]}/{date[1]}/{date[2]} and URL: {custom_url}")
                        else:
                            print("No date information available")
                        break
                
                page_index += 1
                key_index += 1
                pages_processed += 1
                
            except Exception as e:
                print(f"Error in extract_all_VOATibetan_article_links() on page {page_index}: {e}")
                page_index += 1  # Move to next page even after error
                pages_processed += 1
                continue
                
        return_file["Data"] = All_url_links

        pbar.close()
        return return_file
            
    except Exception as e:
        return_file["Data"] = All_url_links
        return_file["message"] = str(e)  # Convert exception to string
        return_file["response"] = 404
        pbar.close()
        return return_file

In [17]:
def check_error_in_links(all_url_links, page_code, print_each_error=False):
    """
    Check for non-200 responses in the all_url_links dictionary
    and count the errors found.
    """
    error_counter = 0
    
    # Iterate through each page entry in the dictionary
    for page_name, page_data in all_url_links.items():
        try:
            if page_data["Response"] != 200:
                error_counter += 1
                if print_each_error:
                    print(f"Error in {page_name}: Response {page_data['Response']}")
        except Exception as e:
            print(f"Exception in {page_name}: {e}")
    
    print(f"Total error in {page_code}: {error_counter}")


In [11]:
def save_json(path, file_name, data):
    """
    
    """
    with open(path+file_name, "w") as outfile:
        json.dump(data, outfile, indent=4)
        print(f"Successfully saved: {file_name}")

In [12]:
def compare_with_existing_data(new_data, existing_file_path, tag):
    """
    Compare newly extracted links with existing data to find new articles.
    
    Args:
        new_data (dict): Dictionary containing newly extracted article links
        existing_file_path (str): Path to the existing JSON file
        tag (str): Tag/category of the articles (e.g., "གོང་ས་མཆོག")
        
    Returns:
        dict: Dictionary containing statistics and new article links
    """
    comparison_result = {
        "tag": tag,
        "total_new_links": 0,
        "total_existing_links": 0,
        "new_links": [],
        "message": "Success",
        "response": 200
    }
    
    try:
        # Load existing data
        with open(existing_file_path, 'r', encoding='utf-8') as file:
            existing_data = json.load(file)
        
        # print(existing_data)
        
        # Extract all existing links into a set for faster lookup
        existing_links = set()
        for page_key in existing_data:
            # print(page_key)
            page_links = existing_data[page_key].get("Links", [])
            for link in page_links:
                existing_links.add(link)
        
        comparison_result["total_existing_links"] = len(existing_links)
        
        # Find new links
        new_links = []
        for page_key in new_data.get("Data", {}):
            page_links = new_data["Data"][page_key].get("Links", [])
            for link in page_links:
                if link not in existing_links:
                    new_links.append(link)
        
        comparison_result["total_new_links"] = len(new_links)
        comparison_result["new_links"] = new_links
        
        return comparison_result
    
    except Exception as e:
        comparison_result["message"] = f"Error comparing data: {str(e)}"
        comparison_result["response"] = 500
        return comparison_result

------------
------------
------------



# B. Extracting all Article links from ཨ་རི། 
- Base url: https://www.voatibetan.com/z/2252?p=1
- Custom URL: https://www.voatibetan.com/z/2252?p= + str(i) 
- Total page: unknown

In [13]:
total_page = 200000 # for custom check
custom_url= "https://www.voatibetan.com/z/2252?p="
article_tag = "ཨ་རི།"
key_code = "Page " + article_tag + " "
print(f"Page code: {key_code}")

all_links = loop_article_page(total_page, custom_url, key_code)


Page code: Page ཨ་རི། 


  0%|          | 102/200000 [08:43<288:16:37,  5.19s/it]

Last article date: 2023-4-1
Moving to new date range: https://www.voatibetan.com/z/2253/2023/4/1?p=


  0%|          | 202/200000 [17:30<286:50:46,  5.17s/it]

Last article date: 2021-6-1
Moving to new date range: https://www.voatibetan.com/z/2253/2021/6/1?p=


...

Last article date: 2011-5-11
Moving to new date range: https://www.voatibetan.com/z/2253/2011/5/11?p=


  0%|          | 870/200000 [1:15:51<294:07:04,  5.32s/it]

Last article date: 1997-2-27
Moving to new date range: https://www.voatibetan.com/z/2253/1997/2/27?p=


  0%|          | 870/200000 [1:15:57<289:45:18,  5.24s/it]

Final page number: 1
Total pages extracted: 869
No date information available


In [14]:
print(f"Total page in {article_tag}: {len(all_links['Data'])}")

Total page in ཨ་རི།: 870


In [18]:
check_error_in_links(all_links['Data'], key_code, print_each_error=True)

Total error in Page ཨ་རི། : 0


In [16]:
# Saving the final file
path = "./new_data/"
file_name = f"VOATibetan_ALL_link_{article_tag}.json"
save_json(path, file_name, all_links['Data'])

Successfully saved: VOATibetan_ALL_link_ཨ་རི།.json
